# 04_Model_editing

In [ ]:
# Configuration -------------------------------------------------------------
import os, json, math, itertools, inspect, random
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

FINE_LOCALIZATION_CSV = run_path("CrisisLexT26_FiNE_ripple_localization.csv")
RIPPLE_SETS_CSV       = run_path("CrisisLexT26_ripple_sets.csv")
PROTECT_SETS_CSV      = run_path("CrisisLexT26_protect_sets.csv")
PREDICTION_CSV_CANDIDATES = [run_path("CrisisLexT26_one_shot_domain_predictions.csv"),
                             run_path("CrisisLexT26_all_predictions_per_instance.csv")]
MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"
ONESHOT_PROMPT, ITERATION, WRONG_TOKEN = "domain", 0, "__wrong__"

# Group-level FiNE selection.
FINE_MASK_MODE      = "member_fraction"   # "member_fraction" (default) or "union" (explicit opt-in)
MIN_MEMBER_FRACTION = 0.5

# Low-level editor (MEND-style) hyperparameters.
EDITOR_RANK, EDITOR_BLOCKS, EDIT_INIT_SCALE = 128, 2, 1e-3   # [FIX] init 1e-2 -> 1e-3 (start tiny)
# [FIX] Descent init. A knowledge edit must REDUCE the edit loss, so the editor's initial update
# must point along -grad. At B=0 init the MEND net returns LayerNorm-normalized (u,v), so
# outer(v,u) is proportional to +∇_W(NLL) — i.e. gradient ASCENT, which pushes the target token
# LESS likely. That wrong sign (plus a too-large magnitude) is what drove the collapse and kept
# ripple accuracy at 0.00->0.00. EDIT_SIGN=-1.0 makes the initial edit a proper descent step;
# training then refines direction and magnitude from a correct starting point.
EDIT_SIGN      = -1.0

# Adapted low-level objective + RL trajectory (groups as timesteps).
LAMBDA_PROTECT = 1.0
ETA            = 1e-2       # [FIX] 1e-3 -> 1e-2 : stronger ||ΔW||^2 regularization (HiEdit Eq.4 η)
MU             = 0.9
MEMORY_K       = 2         # re-evaluate recent previous groups [t-k, t]
GAMMA          = 1.0
LR             = 1e-3       # [FIX] 1e-4 -> 1e-3 : the hypernetwork must actually train
TRAIN_STEPS    = 60        # [FIX] 10 -> 60 : ~10 opt steps was far too few to learn u,v -> ũ,ṽ
# [FIX] Locality guarantee. HiEdit and FiNE both depend on SMALL, localized edits (that is what
# preserves Specificity/locality in the papers). This caps the cumulative
# ||ΔW_sel|| / ||W0_sel|| per edited layer-group so the classifier can NEVER collapse, whatever
# the training dynamics. Raise toward ~0.3 for more edit strength (more targets flipped, less
# locality); lower for more locality. Set to None to disable.
EDIT_CAP       = 0
MAX_PROTECT_PER_GROUP = 5
MAX_GROUPS     = None      # limit #groups for a quick run (None = all)

# A-GEM (Chaudhry et al., ICLR 2019) — added as a gradient-preservation CONSTRAINT on H_theta,
# projecting the accumulated HiEdit trajectory gradient ONCE before the single optimizer.step().
USE_AGEM       = True
AGEM_REF_BATCH = 64        # uniformly sampled protect examples from PREVIOUS-group memory for g_ref
AGEM_REF_CHUNK = 8         # [OPT] chunk size for g_ref accumulation (MEMORY ONLY; does NOT change the
                           # reference size K=AGEM_REF_BATCH). Set >= AGEM_REF_BATCH for the exact
                           # original single-graph reduction.

OUT_LOG = run_path("CrisisLexT26_FiNE_HiEdit_sequential_log.csv")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print(f"device={DEVICE} dtype={DTYPE} | mask={FINE_MASK_MODE}({MIN_MEMBER_FRACTION}) "
      f"K={MEMORY_K} mu={MU} lambda={LAMBDA_PROTECT} eta={ETA} train_steps={TRAIN_STEPS}")

# [OPT] Lightweight GPU-memory probe for the expensive stages (no-op on CPU).
def gpu_mem(tag=""):
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated()/1024**3
        r = torch.cuda.memory_reserved()/1024**3
        mx = torch.cuda.max_memory_allocated()/1024**3
        print(f"[gpu-mem] {tag:34s} alloc={a:6.2f}GiB reserved={r:6.2f}GiB max_alloc={mx:6.2f}GiB")


In [ ]:
# Load FiNE + Ripple/Protect; build the GROUP trajectory --------------------
fine = pd.read_csv(FINE_LOCALIZATION_CSV, dtype=str, keep_default_na=False)
for c in ("layer_id","neuron_id"): fine[c] = fine[c].astype(int)
ripple_sets  = pd.read_csv(RIPPLE_SETS_CSV,  dtype=str, keep_default_na=False)
protect_sets = pd.read_csv(PROTECT_SETS_CSV, dtype=str, keep_default_na=False) \
               if os.path.exists(PROTECT_SETS_CSV) else pd.DataFrame(columns=["sample_id"])
pred_path = next((p for p in PREDICTION_CSV_CANDIDATES if os.path.exists(p)), None)
assert pred_path is not None, "need a predictions CSV for exact prompts"
pred = pd.read_csv(pred_path, dtype=str, keep_default_na=False)
scope = pred.copy()
for c,v in [("shot","one"),("prompt",ONESHOT_PROMPT),("iteration",str(ITERATION))]:
    if c in scope.columns: scope = scope[scope[c].astype(str)==v]
meta = scope.drop_duplicates("sample_id").set_index("sample_id")
def _has_meta(sid): return sid in meta.index and bool(str(meta.loc[sid].get("exact_messages_json","")))

def _group_mask(gid):
    g = fine[fine["ripple_group_id"].astype(str)==str(gid)]
    members = [s for s in g["sample_id"].unique() if _has_meta(s)]
    if not members: return {}, members
    per_member = {s:set() for s in members}
    for _, r in g.iterrows():
        s = str(r["sample_id"])
        if s in per_member: per_member[s].add((int(r["layer_id"]), int(r["neuron_id"])))
    if FINE_MASK_MODE == "union":
        chosen = set().union(*per_member.values())
    elif FINE_MASK_MODE == "member_fraction":
        cnt = Counter()
        for s in members:
            for ln in per_member[s]: cnt[ln] += 1
        chosen = {ln for ln, c in cnt.items() if c/len(members) >= MIN_MEMBER_FRACTION}
    else:
        raise ValueError(f"unknown FINE_MASK_MODE {FINE_MASK_MODE!r}")
    sel = defaultdict(list)
    for (l,n) in chosen: sel[l].append(n)
    return {l: sorted(v) for l,v in sel.items()}, members

# GLOBAL Protect memory = ALL W0-correct samples (one pool, shared by every group).
# No per-group / ripple_group_id / same-class / top-k / MAX_PROTECT_PER_GROUP selection
# is applied here; membership is exactly the global Protect set built in Section 12.
GLOBAL_PROTECT_IDS = [s for s in protect_sets["sample_id"].astype(str).unique() if _has_meta(s)]
print(f"[protect] GLOBAL Protect memory: {len(GLOBAL_PROTECT_IDS)} W0-correct samples with metadata "
      f"(of {protect_sets['sample_id'].nunique()} in CSV)")

group_ids = sorted(fine["ripple_group_id"].astype(str).unique(),
                   key=lambda x:(int(x) if x.isdigit() else 1_000_000, x))
groups = []
for gid in group_ids:
    sel_by_layer, members = _group_mask(gid)
    if not members or not sel_by_layer:      # no members with metadata, or no neuron >= fraction
        print(f"  [skip] group {gid}: members={len(members)} editable_layers={len(sel_by_layer)}")
        continue
    tl = ripple_sets[ripple_sets["sample_id"].astype(str)==str(members[0])]["true_label"]
    groups.append({"gid": gid, "member_ids": members, "protect_ids": GLOBAL_PROTECT_IDS,
                   "sel_by_layer": sel_by_layer, "true_label": tl.iloc[0] if len(tl) else "?"})
if MAX_GROUPS is not None: groups = groups[:MAX_GROUPS]
assert groups, "no editable groups"

ALL_EDITED_LAYERS = sorted(set(l for g in groups for l in g["sel_by_layer"]))
print(f"trajectory: {len(groups)} groups | edited layers (union): {ALL_EDITED_LAYERS}")
print(f"GLOBAL Protect memory shared by all groups: {len(GLOBAL_PROTECT_IDS)} samples")
for t,g in enumerate(groups):
    nsel = sum(len(v) for v in g["sel_by_layer"].values())
    print(f"  t={t} group {g['gid']} | members={len(g['member_ids'])} protect={len(g['protect_ids'])} "
          f"| layers={list(g['sel_by_layer'])} neurons={nsel} | true_label={g['true_label']}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM

# [OPT] Free the FiNE-localization-stage model/tensors before the editor stage loads its own copy,
# so two Qwen2.5-3B models never sit on the GPU at once. Memory management only: the editor stage
# reloads the identical weights below, so nothing about the pipeline/method changes. Guarded so it
# is a no-op when this stage is run in a fresh kernel where these names do not exist.
import gc as _gc
try:
    for _h in _hook_handles: _h.remove()      # remove FiNE down_proj capture pre-hooks
except Exception:
    pass
for _name in ["_hook_handles","_capture","model","layers","W_u","W_out0","mlp0","q_by_layer"]:
    if _name in globals():
        try: del globals()[_name]
        except Exception: pass
_gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# Load Qwen2.5-3B, verify down_proj orientation, per-group column masks -----
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE).to(DEVICE).eval()
cfg = model.config
d_h, d_m, NLAYERS, VOCAB = cfg.hidden_size, cfg.intermediate_size, cfg.num_hidden_layers, cfg.vocab_size
layers = model.model.layers
W_dp0 = layers[ALL_EDITED_LAYERS[0]].mlp.down_proj.weight
assert tuple(W_dp0.shape)==(d_h,d_m), f"down_proj.weight {tuple(W_dp0.shape)} != (d_h,d_m)"
print(f"down_proj.weight = [d_h,d_m] = [{d_h},{d_m}] => FiNE neuron i is COLUMN i  [asserted]")

def pname(l): return f"model.layers.{l}.mlp.down_proj.weight"
for g in groups:
    for l, ns in g["sel_by_layer"].items():
        for n in ns: assert 0 <= n < d_m
    # per-group per-layer column mask
    g["col_mask"] = {l: torch.zeros(1, d_m, device=DEVICE, dtype=torch.float32) for l in g["sel_by_layer"]}
    for l, ns in g["sel_by_layer"].items(): g["col_mask"][l][0, ns] = 1.0
    g["edited_layers"] = sorted(g["sel_by_layer"].keys())

# pristine W0 for every edited layer + freeze ALL params (rule: local patch, not weight overrides)
W0 = {l: layers[l].mlp.down_proj.weight.detach().clone() for l in ALL_EDITED_LAYERS}
MODEL_PARAM_DTYPE = next(model.parameters()).dtype
for p in model.parameters(): p.requires_grad_(False)
def _pdig(pt):
    d = pt.detach();  return (float(d.double().sum()), float((d.double()**2).sum()))
PARAM_DIGEST_BEFORE = {n:_pdig(pp) for n,pp in model.named_parameters()}
print(f"all params frozen; W0 stored for {len(W0)} layers; baseline digest for "
      f"{len(PARAM_DIGEST_BEFORE)} tensors")
gpu_mem("editor stage: model loaded")


In [ ]:
# Exact prompt reconstruction + contextual target tokenization -------------
def reconstruct_prompt(sid):
    obj = json.loads(meta.loc[sid, "exact_messages_json"])
    prompt_str = tokenizer.apply_chat_template(obj, tokenize=False, add_generation_prompt=True) \
                 if isinstance(obj, list) else str(obj)
    ids = tokenizer(prompt_str, add_special_tokens=False, return_tensors="pt").input_ids
    return prompt_str, ids

def contextual_label_ids(prompt_str, label_text):
    p = tokenizer(prompt_str, add_special_tokens=False)["input_ids"]
    f = tokenizer(prompt_str + label_text, add_special_tokens=False)["input_ids"]
    cpl = 0
    for a,b in zip(p,f):
        if a==b: cpl+=1
        else: break
    return f[cpl:]

def build_example(sid, target_label):
    ps, pid = reconstruct_prompt(sid)
    lab = contextual_label_ids(ps, target_label)
    full = torch.cat([pid, torch.tensor([lab])], dim=1).to(DEVICE)
    return {"sample_id": sid, "prompt_len": pid.shape[1], "full_ids": full,
            "label_ids": lab, "target": target_label}

def _true_label(sid):
    r = ripple_sets[ripple_sets["sample_id"].astype(str)==str(sid)]
    return r.iloc[0]["true_label"] if len(r) else "?"

def _protect_label(sid):
    pr = protect_sets[protect_sets["sample_id"].astype(str)==str(sid)].iloc[0]
    return pr["true_label"] if ("true_label" in pr and pr["true_label"]) else pr.get("predicted_label","")

# Build the ONE global Protect memory of prepared examples ONCE (ALL W0-correct
# samples), then share it (read-only) with every group. Protect membership is global —
# not per-group / same-class / top-k. Cell 8 uses this global list directly for the
# full-memory Protect KL and A-GEM; the per-group diagnostics in Cell 9 read the same pool.
GLOBAL_PROTECT = [build_example(s, _protect_label(s)) for s in GLOBAL_PROTECT_IDS]

for g in groups:
    g["members"] = [build_example(s, _true_label(s)) for s in g["member_ids"]]
    g["protect"] = GLOBAL_PROTECT          # shared GLOBAL Protect memory (read-only)

print("built examples for", len(groups), "groups "
      f"({sum(len(g['members']) for g in groups)} ripple, "
      f"{len(GLOBAL_PROTECT)} global protect [shared])")


In [ ]:
# Metric helpers (base-model forward; no functional_call) ------------------
@torch.no_grad()
def base_logits(input_ids):
    return model(input_ids=input_ids, use_cache=False).logits

def label_positions(prompt_len, n_label):
    return torch.arange(prompt_len-1, prompt_len-1+n_label, device=DEVICE)

@torch.no_grad()
def eval_example(ex):
    logits = base_logits(ex["full_ids"])
    pos = label_positions(ex["prompt_len"], len(ex["label_ids"]))
    lp = F.log_softmax(logits[0,pos].float(), -1)
    tgt = torch.tensor(ex["label_ids"], device=DEVICE)
    tl = lp[torch.arange(len(tgt)), tgt]
    return {"mean_tok_prob": tl.exp().mean().item(),
            "top1": (lp.argmax(-1)==tgt).float().mean().item(),
            "exact": bool((lp.argmax(-1)==tgt).all().item())}

@torch.no_grad()
def answer_distribution(ex):
    logits = base_logits(ex["full_ids"][:, :ex["prompt_len"]])
    return F.softmax(logits[0, ex["prompt_len"]-1].float(), -1)

def acc_over(examples):
    if not examples: return {"exact": float("nan"), "top1": float("nan"), "n": 0}
    r = [eval_example(e) for e in examples]
    return {"exact": float(np.mean([x["exact"] for x in r])),
            "top1":  float(np.mean([x["top1"]  for x in r])), "n": len(examples)}


In [ ]:
# Low-level HiEdit editor H_theta (MEND-style, Eq. 8) — UNCHANGED ----------
class LowLevelEditor(nn.Module):
    def __init__(self, d_m, d_h, layer_ids, d_r=128, C=2, init_scale=1e-2):
        super().__init__()
        self.d_m, self.d_h, self.C = d_m, d_h, C
        d = d_m + d_h
        self.in_norm = nn.LayerNorm(d)
        self.A = nn.ModuleList([nn.Linear(d, d_r, bias=False) for _ in range(C)])
        self.B = nn.ModuleList([nn.Linear(d_r, d, bias=False) for _ in range(C)])
        for b in self.B: nn.init.zeros_(b.weight)
        self.scale, self.offset = nn.ParameterDict(), nn.ParameterDict()
        for l in layer_ids:
            for c in range(C):
                self.scale[f"{l}_{c}"]  = nn.Parameter(torch.ones(d))
                self.offset[f"{l}_{c}"] = nn.Parameter(torch.zeros(d))
        self.log_edit_scale = nn.Parameter(torch.log(torch.tensor(float(init_scale))))
    def forward(self, u, v, layer_id):
        h = self.in_norm(torch.cat([u, v], dim=-1))
        for c in range(self.C):
            z = self.B[c](F.relu(self.A[c](h)))
            z = self.scale[f"{layer_id}_{c}"] * z + self.offset[f"{layer_id}_{c}"]
            h = h + z
        return h[..., :self.d_m], h[..., self.d_m:]
    def delta_w(self, u, v, layer_id, col_mask):
        ut, vt = self(u, v, layer_id)
        # [FIX] EDIT_SIGN=-1.0 makes the initial (untrained) update a DESCENT step on the edit
        # loss (−∇_W NLL) instead of ascent. Everything else (H_θ, rank-1 ṽũᵀ, FiNE mask) is HiEdit.
        dW = EDIT_SIGN * torch.exp(self.log_edit_scale) * torch.outer(vt, ut)
        return dW * col_mask

editor = LowLevelEditor(d_m, d_h, ALL_EDITED_LAYERS, d_r=EDITOR_RANK, C=EDITOR_BLOCKS,
                        init_scale=EDIT_INIT_SCALE).to(DEVICE).float()
print(f"editor params: {sum(p.numel() for p in editor.parameters()):,}")


In [ ]:
# LOCAL down_proj patch + rank-1 feature extraction — UNCHANGED mechanics --
_edit_handles = []
def install_edit_patch(delta_map):
    """down_proj output -> base_out + F.linear(x, delta_l); delta_l differentiable (grad->θ)."""
    remove_edit_patch()
    for l in ALL_EDITED_LAYERS:
        dp = layers[l].mlp.down_proj
        def mk(l):
            def hook(module, args, output):
                d = delta_map.get(l, None)
                if d is None: return output
                x = args[0]
                return output + F.linear(x, d.to(dtype=x.dtype, device=x.device), None)
            return hook
        _edit_handles.append(dp.register_forward_hook(mk(l)))

def remove_edit_patch():
    while _edit_handles: _edit_handles.pop().remove()

def compute_uv(ex, delta_state, feature_layers):
    """Rank-1 (u_l,v_l) at the label position under W_{t-1} (= W0 + delta_state, injected).
    Injects accumulated edits on ALL_EDITED_LAYERS; captures features on feature_layers only."""
    for l in feature_layers: layers[l].mlp.down_proj.weight.requires_grad_(True)
    inp, out, handles = {}, {}, []
    for l in ALL_EDITED_LAYERS:
        dp = layers[l].mlp.down_proj; d_l = delta_state.get(l, None); cap = (l in feature_layers)
        def mk(l, d_l, cap):
            def hook(m, args, output):
                x = args[0]; o = output
                if d_l is not None:
                    o = o + F.linear(x, d_l.to(dtype=x.dtype, device=x.device), None)
                if cap: inp[l] = x; out[l] = o
                return o
            return hook
        handles.append(dp.register_forward_hook(mk(l, d_l, cap)))
    with torch.enable_grad():
        logits = model(input_ids=ex["full_ids"], use_cache=False).logits
        pos = label_positions(ex["prompt_len"], len(ex["label_ids"]))
        lp = F.log_softmax(logits[0,pos].float(), -1)
        tgt = torch.tensor(ex["label_ids"], device=DEVICE)
        L = -lp[torch.arange(len(tgt)), tgt].mean()
        grads = torch.autograd.grad(L, [out[l] for l in feature_layers], retain_graph=False)
    for h in handles: h.remove()
    for l in feature_layers: layers[l].mlp.down_proj.weight.requires_grad_(False)
    ps = ex["prompt_len"] - 1
    return {l: (inp[l][0,ps].detach().float(), g[0,ps].detach().float())
            for l, g in zip(feature_layers, grads)}

def restore_all_W0():
    with torch.no_grad():
        for l in ALL_EDITED_LAYERS: layers[l].mlp.down_proj.weight.copy_(W0[l])





def cap_group_delta(gdW, group):
    """
    Enforce the actual cumulative locality constraint:

        || (W_current - W0) + alpha * dW || <= EDIT_CAP * ||W0||

    on this group's FiNE-selected columns.

    Unlike the old implementation, this uses the real vector geometry
    instead of the conservative approximation:
        ||C + D|| <= ||C|| + ||D||
    """

    if EDIT_CAP is None or EDIT_CAP <= 0:
        return gdW

    c2 = 0.0      # ||C||^2
    d2 = 0.0      # ||D||^2
    cd = 0.0      # <C,D>
    w02 = 0.0     # ||W0||^2 on selected columns

    for l in group["edited_layers"]:

        idx = torch.tensor(
            group["sel_by_layer"][l],
            device=W0[l].device,
            dtype=torch.long
        )

        current = (
            layers[l]
            .mlp.down_proj.weight
            .detach()
            .float()
            .index_select(1, idx)
        )

        base = (
            W0[l]
            .detach()
            .float()
            .index_select(1, idx)
        )

        # cumulative displacement before this group
        C = current - base

        # proposed update on the same selected columns
        D = (
            gdW[l]
            .detach()
            .float()
            .index_select(1, idx)
        )

        c2  += float((C * C).sum())
        d2  += float((D * D).sum())
        cd  += float((C * D).sum())
        w02 += float((base * base).sum())

    cur_norm = math.sqrt(max(c2, 0.0))
    gnorm    = math.sqrt(max(d2, 0.0))
    budget   = EDIT_CAP * math.sqrt(max(w02, 0.0))

    print(
        f"[CAP DETAIL] group={group['gid']} "
        f"cur_norm={cur_norm:.6f} "
        f"budget={budget:.6f} "
        f"gnorm={gnorm:.6f} "
        f"dot={cd:.6f}"
    )

    if d2 <= 0.0:
        print("[CAP DETAIL] proposed update already zero")
        return gdW

    # What happens if we apply the full update?
    final2_full = c2 + 2.0 * cd + d2

    if final2_full <= budget * budget:
        alpha = 1.0

    else:
        # Solve exactly:
        #
        # ||C + alpha D||^2 <= budget^2
        #
        # d2*alpha^2 + 2*cd*alpha
        # + (c2-budget^2) <= 0

        A = d2
        B = 2.0 * cd
        Cq = c2 - budget * budget

        disc = B * B - 4.0 * A * Cq

        if disc < 0.0:
            # Proposed direction never enters the allowed ball.
            alpha = 0.0

        else:
            sqrt_disc = math.sqrt(max(disc, 0.0))

            r1 = (-B - sqrt_disc) / (2.0 * A)
            r2 = (-B + sqrt_disc) / (2.0 * A)

            lo = min(r1, r2)
            hi = max(r1, r2)

            # Feasible interval along alpha.
            left  = max(0.0, lo)
            right = min(1.0, hi)

            if left <= right:
                # largest feasible amount of the proposed edit
                alpha = right
            else:
                alpha = 0.0

    print(f"[CAP DETAIL] alpha={alpha:.6f}")

    return {
        l: gdW[l] * alpha
        for l in gdW
    }


In [ ]:
# Group-level ΔW, aggregated NLL, and W_{t-1}-relative Protect KL -----------
def make_group_delta(group, delta_state):
    """One group-level ΔW per edited layer = MEAN over members of masked editor.delta_w (differentiable).
    ΔW_group = (1/|G|) Σ_i ΔW_i. Divided exactly ONCE here; train_editor() must NOT divide again.

    [OPT] Memory-only rewrite, BIT-IDENTICAL to the original in value AND gradient. The original
    built one dense [d_h, d_m] masked outer product per member and kept them all in the autograd
    graph until the group backward (n_members × n_layers × ~90MB -> the OOM driver for whole-class
    Ripple groups). Since outer(v,u)⊙col_mask keeps only the FiNE-selected columns, we accumulate
    just those columns ([d_h, S]) per member, in the SAME member order, with the SAME per-member
    scalar EDIT_SIGN·exp(log_edit_scale), then scatter the /|G| mean into a zero [d_h, d_m]. Because
    outer(v,u)[:,sel] == outer(v, u[sel]) and x·1.0 == x, every stored value and its gradient match
    the original exactly; only structurally-zero columns are skipped. Return type is unchanged."""
    n_members = len(group["members"])
    assert n_members > 0, "make_group_delta: empty group"
    els = group["edited_layers"]
    sel_idx = {l: group["col_mask"][l][0].nonzero(as_tuple=False).flatten() for l in els}
    acc = {l: None for l in els}
    for ex in group["members"]:
        uv = compute_uv(ex, delta_state, els)
        for l in els:
            ut, vt = editor(uv[l][0], uv[l][1], l)                # same H_θ forward as editor.delta_w
            ut_sel = ut.index_select(0, sel_idx[l])               # selected u-entries only ([S])
            contrib = EDIT_SIGN * torch.exp(editor.log_edit_scale) * torch.outer(vt, ut_sel)  # [d_h,S]
            acc[l] = contrib if acc[l] is None else acc[l] + contrib
    dW = {}
    for l in els:                     # MEAN group action: divide exactly once by |G|; scatter to dense
        full = torch.zeros(d_h, d_m, dtype=acc[l].dtype, device=acc[l].device)
        dW[l] = full.index_copy(1, sel_idx[l], acc[l] / n_members)
    return dW

def group_mean_nll(group):
    """Mean NLL(true_label) over a group's members, under whatever patch/weights are active."""
    tot = 0.0
    for ex in group["members"]:
        logits = model(input_ids=ex["full_ids"], use_cache=False).logits
        pos = label_positions(ex["prompt_len"], len(ex["label_ids"]))
        lp = F.log_softmax(logits[0,pos].float(), -1)
        tgt = torch.tensor(ex["label_ids"], device=DEVICE)
        tot = tot + (-lp[torch.arange(len(tgt)), tgt].mean())
    return tot / len(group["members"])

def protect_kl(protect_list, P_before):
    """KL[P_before(=P_{W_{t-1}}) || P_after(=current patch/weights)] averaged over the group's protect set."""
    if not protect_list: return torch.zeros((), device=DEVICE)
    tot = 0.0
    for e, Pb in zip(protect_list, P_before):
        logits = model(input_ids=e["full_ids"][:, :e["prompt_len"]], use_cache=False).logits
        logPa = F.log_softmax(logits[0, e["prompt_len"]-1].float(), -1)
        tot = tot + F.kl_div(logPa, Pb, reduction="sum")
    return tot / len(protect_list)


In [ ]:
import inspect
import random
import torch


def _merge_state_delta(delta_state, gdW):
    dm = {}

    for l in set(delta_state) | set(gdW):
        base = delta_state.get(l, None)
        cur = gdW.get(l, None)

        if cur is None:
            dm[l] = base

        elif base is None:
            dm[l] = cur

        else:
            dm[l] = base.to(cur.dtype) + cur

    return dm


def nll_example(ex):
    logits = model(
        input_ids=ex["full_ids"],
        use_cache=False
    ).logits

    pos = label_positions(
        ex["prompt_len"],
        len(ex["label_ids"])
    )

    lp = F.log_softmax(
        logits[0, pos].float(),
        dim=-1
    )

    tgt = torch.tensor(
        ex["label_ids"],
        device=DEVICE
    )

    return -lp[
        torch.arange(len(tgt)),
        tgt
    ].mean()


def kl_example(ex, P_before):
    logits = model(
        input_ids=ex["full_ids"][:, :ex["prompt_len"]],
        use_cache=False
    ).logits

    logP_after = F.log_softmax(
        logits[0, ex["prompt_len"] - 1].float(),
        dim=-1
    )

    return F.kl_div(
        logP_after,
        P_before,
        reduction="sum"
    )


def _agem_sample(memory, k):
    return (
        list(memory)
        if len(memory) <= k
        else random.sample(memory, k)
    )


In [ ]:
# ============================================================
# Cell 8 — FAST-EQUIVALENT HiEdit trajectory training
#          + FULL Protect-Memory KL
#          + FULL Protect-Memory A-GEM
#
# EXECUTION-ONLY OPTIMIZATION + CHECKPOINT/RESUME — SCIENTIFIC LOGIC UNCHANGED
#
# PRESERVED EXACTLY:
#   - FiNE localization
#   - HiEdit low-level editor
#   - make_group_delta()
#   - selected FiNE neurons / columns
#   - group action MEAN exactly once
#   - sequential W0 -> W1 -> ... -> WT trajectory
#   - detached delta_state transition
#   - edit loss + memory backtracking
#   - MEMORY_K / MU / GAMMA weighting
#   - FULL GLOBAL Protect memory
#   - ALL Protect samples used for KL at EVERY group
#   - KL(P_before || P_after)
#   - /N_PROTECT_FULL normalization
#   - ETA ||DeltaW_t||^2
#   - FULL Protect-memory A-GEM
#   - AGEM_REF_BATCH unchanged
#   - AGEM_REF_CHUNK unchanged in scientific meaning
#   - A-GEM reference objective unchanged
#   - A-GEM projection rule unchanged
#   - one optimizer.step() per COMPLETE trajectory
#   - optimizer / LR / clipping / TRAIN_STEPS unchanged
#   - checkpoint/history persistence
#
# SPEED CHANGES ONLY:
#   1) NLL forwards microbatched
#   2) Full-Protect P_before forwards microbatched
#   3) Full-Protect KL forwards microbatched
#   4) A-GEM forward inside each ORIGINAL AGEM_REF_CHUNK is
#      internally microbatched, but the original chunk objective
#      and denominator K are preserved exactly
#   5) no torch.cuda.empty_cache() inside every group
#
# IMPORTANT:
#   All execution microbatches are set to 4.
#   This does NOT drop samples or change the objective.
# ============================================================

import random
import inspect
import os
import time
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd


# ============================================================
# EXECUTION-ONLY microbatch sizes
# ============================================================

FAST_NLL_BATCH = 16
FAST_KL_BATCH = 16
FAST_BEFORE_BATCH = 16
FAST_AGEM_MICROBATCH = 16   # execution only; AGEM_REF_CHUNK itself is NOT changed


# ============================================================
# CHECKPOINT / RESUME — EXECUTION ONLY
#
# Training state is saved ONLY between complete trajectories,
# after opt.step(). Therefore a resumed run starts at the next
# complete trajectory and does not change the HiEdit trajectory
# computation inside any TRAIN_STEP.
# ============================================================

RESUME_TRAINING = True
CHECKPOINT_EVERY = 1

TRAIN_CHECKPOINT_PATH = run_path(
    "cell8_training_checkpoint.pt"
)


def _capture_rng_state():

    state = {
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_cpu_rng_state": torch.get_rng_state(),
    }

    if torch.cuda.is_available():
        state["torch_cuda_rng_state_all"] = (
            torch.cuda.get_rng_state_all()
        )

    return state


def _restore_rng_state(ckpt):

    if "python_random_state" in ckpt:
        random.setstate(
            ckpt["python_random_state"]
        )

    if "numpy_random_state" in ckpt:
        np.random.set_state(
            ckpt["numpy_random_state"]
        )

    if "torch_cpu_rng_state" in ckpt:
        torch.set_rng_state(
            ckpt["torch_cpu_rng_state"]
        )

    if (
        torch.cuda.is_available()
        and
        "torch_cuda_rng_state_all" in ckpt
    ):
        torch.cuda.set_rng_state_all(
            ckpt["torch_cuda_rng_state_all"]
        )


def _atomic_torch_save(obj, path):

    tmp_path = path + ".tmp"

    torch.save(
        obj,
        tmp_path
    )

    os.replace(
        tmp_path,
        path
    )


# ============================================================
# CUDA helper
# ============================================================

_IS_CUDA = (
    torch.cuda.is_available()
    and str(DEVICE).startswith("cuda")
)


# ============================================================
# Existing helper:
# W0 + detached previous state + current differentiable action
# ============================================================

def _merge_state_delta(delta_state, gdW):

    dm = {}

    for l in set(delta_state) | set(gdW):

        base = delta_state.get(l, None)
        cur  = gdW.get(l, None)

        if cur is None:
            dm[l] = base

        elif base is None:
            dm[l] = cur

        else:
            dm[l] = (
                base.to(cur.dtype)
                + cur
            )

    return dm


# ============================================================
# Small execution helpers
# ============================================================

def _chunks(seq, batch_size):

    batch_size = max(
        1,
        int(batch_size)
    )

    for s in range(
        0,
        len(seq),
        batch_size
    ):
        yield seq[s:s + batch_size]


def _pad_right(id_tensors):

    assert len(id_tensors) > 0

    seqs = [
        x.reshape(-1).to(DEVICE)
        for x in id_tensors
    ]

    lengths = [
        int(x.numel())
        for x in seqs
    ]

    max_len = max(lengths)

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    if pad_id is None:
        raise RuntimeError(
            "Tokenizer has neither pad_token_id nor eos_token_id."
        )

    input_ids = torch.full(
        (len(seqs), max_len),
        int(pad_id),
        dtype=seqs[0].dtype,
        device=DEVICE
    )

    attention_mask = torch.zeros(
        (len(seqs), max_len),
        dtype=torch.long,
        device=DEVICE
    )

    for b, ids in enumerate(seqs):

        L = ids.numel()

        input_ids[b, :L] = ids
        attention_mask[b, :L] = 1

    return input_ids, attention_mask, lengths


# ============================================================
# ORIGINAL single-example NLL
# Kept for compatibility / diagnostics
# ============================================================

def nll_example(ex):

    logits = model(
        input_ids=ex["full_ids"],
        use_cache=False
    ).logits

    pos = label_positions(
        ex["prompt_len"],
        len(ex["label_ids"])
    )

    lp = F.log_softmax(
        logits[0, pos].float(),
        dim=-1
    )

    tgt = torch.tensor(
        ex["label_ids"],
        device=DEVICE
    )

    return -lp[
        torch.arange(
            len(tgt),
            device=DEVICE
        ),
        tgt
    ].mean()


# ============================================================
# Batched NLL: sum of original per-example mean NLLs
# ============================================================

def nll_batch_sum(examples):

    examples = list(examples)

    if len(examples) == 0:

        return torch.zeros(
            (),
            dtype=torch.float32,
            device=DEVICE
        )

    input_ids, attention_mask, _ = _pad_right(
        [ex["full_ids"][0] for ex in examples]
    )

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False
    ).logits

    losses = []

    for b, ex in enumerate(examples):

        pos = label_positions(
            ex["prompt_len"],
            len(ex["label_ids"])
        )

        lp = F.log_softmax(
            logits[b, pos].float(),
            dim=-1
        )

        tgt = torch.tensor(
            ex["label_ids"],
            dtype=torch.long,
            device=DEVICE
        )

        ex_nll = -lp[
            torch.arange(
                len(tgt),
                device=DEVICE
            ),
            tgt
        ].mean()

        losses.append(ex_nll)

    return torch.stack(losses).sum()


# ============================================================
# P_before for FULL Protect memory
# ============================================================

@torch.no_grad()
def protect_distributions_batched(
    memory,
    batch_size=FAST_BEFORE_BATCH
):

    outputs = []

    for batch in _chunks(
        list(memory),
        batch_size
    ):

        prompt_seqs = [
            ex["full_ids"][
                0,
                :ex["prompt_len"]
            ]
            for ex in batch
        ]

        input_ids, attention_mask, _ = _pad_right(
            prompt_seqs
        )

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False
        ).logits

        for b, ex in enumerate(batch):

            outputs.append(
                F.softmax(
                    logits[
                        b,
                        ex["prompt_len"] - 1
                    ].float(),
                    dim=-1
                ).detach()
            )

        del logits, input_ids, attention_mask

    return outputs


# ============================================================
# ORIGINAL single-example KL
# ============================================================

def kl_example(ex, P_before):

    prompt_ids = ex["full_ids"][
        :,
        :ex["prompt_len"]
    ]

    logits = model(
        input_ids=prompt_ids,
        use_cache=False
    ).logits

    logP_after = F.log_softmax(
        logits[
            0,
            ex["prompt_len"] - 1
        ].float(),
        dim=-1
    )

    return F.kl_div(
        logP_after,
        P_before,
        reduction="sum"
    )


# ============================================================
# Batched KL: sum of original per-example KLs
# ============================================================

def protect_kl_batch_sum(
    examples,
    P_before
):

    examples = list(examples)
    P_before = list(P_before)

    assert len(examples) == len(P_before)

    if len(examples) == 0:

        return torch.zeros(
            (),
            dtype=torch.float32,
            device=DEVICE
        )

    prompt_seqs = [
        ex["full_ids"][
            0,
            :ex["prompt_len"]
        ]
        for ex in examples
    ]

    input_ids, attention_mask, _ = _pad_right(
        prompt_seqs
    )

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False
    ).logits

    vals = []

    for b, (ex, Pb) in enumerate(
        zip(examples, P_before)
    ):

        logP_after = F.log_softmax(
            logits[
                b,
                ex["prompt_len"] - 1
            ].float(),
            dim=-1
        )

        vals.append(
            F.kl_div(
                logP_after,
                Pb,
                reduction="sum"
            )
        )

    return torch.stack(vals).sum()


# ============================================================
# A-GEM sampling — UNCHANGED
# ============================================================

def _agem_sample(memory, k):

    return (
        list(memory)
        if len(memory) <= k
        else random.sample(memory, k)
    )


# ============================================================
# Train editor
# ============================================================

def train_editor():

    opt = torch.optim.Adam(
        editor.parameters(),
        lr=LR
    )

    params = list(
        editor.parameters()
    )

    history = []

    # --------------------------------------------------------
    # Resume state — ONLY between completed trajectories
    # --------------------------------------------------------

    start_step = 0

    if (
        RESUME_TRAINING
        and
        os.path.exists(
            TRAIN_CHECKPOINT_PATH
        )
    ):

        print(
            f"[RESUME] loading checkpoint: "
            f"{TRAIN_CHECKPOINT_PATH}"
        )

        ckpt = torch.load(
            TRAIN_CHECKPOINT_PATH,
            map_location=DEVICE
        )

        # Guard against accidentally resuming a scientifically
        # different experiment.
        assert ckpt["train_steps"] == TRAIN_STEPS
        assert ckpt["lr"] == LR
        assert ckpt["gamma"] == GAMMA
        assert ckpt["mu"] == MU
        assert ckpt["memory_k"] == MEMORY_K
        assert ckpt["lambda_protect"] == LAMBDA_PROTECT
        assert ckpt["eta"] == ETA
        assert ckpt["use_agem"] == USE_AGEM
        assert ckpt["agem_ref_batch"] == AGEM_REF_BATCH
        assert ckpt["agem_ref_chunk"] == AGEM_REF_CHUNK
        assert ckpt["n_samples"] == N_SAMPLES

        editor.load_state_dict(
            ckpt["editor_state_dict"]
        )

        opt.load_state_dict(
            ckpt["optimizer_state_dict"]
        )

        history = list(
            ckpt["history"]
        )

        start_step = int(
            ckpt["next_step"]
        )

        _restore_rng_state(
            ckpt
        )

        print(
            f"[RESUME] continuing from "
            f"traversal {start_step}/{TRAIN_STEPS}"
        )

        if start_step >= TRAIN_STEPS:

            print(
                "[RESUME] training already complete."
            )

            return history


    def zero_grad_buffer():

        return [
            torch.zeros_like(p)
            for p in params
        ]


    def add_grads_(buffer, grads):

        with torch.no_grad():

            for buf, g in zip(
                buffer,
                grads
            ):

                if g is not None:
                    buf.add_(g.detach())


    def buffer_dot(a, b):

        return float(
            sum(
                (x.float() * y.float()).sum()
                for x, y in zip(a, b)
            )
        )


    def buffer_norm2(a):

        return float(
            sum(
                (x.float() * x.float()).sum()
                for x in a
            )
        )


    full_protect_memory = list(
        GLOBAL_PROTECT
    )

    print(
        f"[Protect] full GLOBAL memory = "
        f"{len(full_protect_memory)} W0-correct samples"
    )

    assert len(full_protect_memory) > 0, (
        "No Protect samples found."
    )

    if USE_AGEM:
        print(
            "[A-GEM] using full Protect memory "
            "from G0 onward"
        )

    print(
        "[KL] FULL Protect-memory KL enabled "
        "at every group"
    )

    print("[FAST] execution-only batching:")
    print(f"  NLL microbatch      = {FAST_NLL_BATCH}")
    print(f"  KL microbatch       = {FAST_KL_BATCH}")
    print(f"  P_before microbatch = {FAST_BEFORE_BATCH}")
    print(f"  A-GEM microbatch    = {FAST_AGEM_MICROBATCH}")
    print(f"  AGEM_REF_BATCH      = {AGEM_REF_BATCH} (UNCHANGED)")
    print(f"  AGEM_REF_CHUNK      = {AGEM_REF_CHUNK} (UNCHANGED)")


    if _IS_CUDA:
        torch.cuda.reset_peak_memory_stats()

    gpu_mem(
        "train_editor: start"
    )


    for step in range(
        start_step,
        TRAIN_STEPS
    ):

        opt.zero_grad(
            set_to_none=True
        )

        delta_state = {}

        protect_memory = list(
            full_protect_memory
        )

        trajectory_grad = (
            zero_grad_buffer()
        )

        J = 0.0
        groups_processed = 0


        for t, group in enumerate(groups):

            gt = GAMMA ** t


            # 1. FULL Protect P_before under W_{t-1}

            install_edit_patch(
                delta_state
            )

            P_before_full = (
                protect_distributions_batched(
                    full_protect_memory,
                    FAST_BEFORE_BATCH
                )
            )

            remove_edit_patch()

            assert (
                len(P_before_full)
                ==
                len(full_protect_memory)
            )


            # 2. Current HiEdit action ΔW_t — UNCHANGED

            gdW = make_group_delta(
                group,
                delta_state
            )

            state_t = _merge_state_delta(
                delta_state,
                gdW
            )

            install_edit_patch(
                state_t
            )


            # 3. Current group gradient

            g_cur = zero_grad_buffer()
            L_val = 0.0


            # 3A. HiEdit edit loss + memory backtracking

            for i in range(
                max(0, t - MEMORY_K),
                t + 1
            ):

                members_i = list(
                    groups[i]["members"]
                )

                w = (
                    MU ** (t - i)
                ) / max(
                    1,
                    len(members_i)
                )

                for batch in _chunks(
                    members_i,
                    FAST_NLL_BATCH
                ):

                    c = (
                        gt
                        * w
                        * nll_batch_sum(batch)
                    )

                    grads = torch.autograd.grad(
                        c,
                        params,
                        retain_graph=True,
                        allow_unused=True
                    )

                    add_grads_(
                        g_cur,
                        grads
                    )

                    L_val += float(
                        c.detach()
                    )

                    del c, grads


            # 3B. FULL Protect-memory KL

            N_PROTECT_FULL = len(
                full_protect_memory
            )

            n_kl_done = 0

            for s in range(
                0,
                N_PROTECT_FULL,
                FAST_KL_BATCH
            ):

                batch_examples = (
                    full_protect_memory[
                        s:s + FAST_KL_BATCH
                    ]
                )

                batch_P_before = (
                    P_before_full[
                        s:s + FAST_KL_BATCH
                    ]
                )

                c = (
                    gt
                    * LAMBDA_PROTECT
                    / N_PROTECT_FULL
                    * protect_kl_batch_sum(
                        batch_examples,
                        batch_P_before
                    )
                )

                grads = torch.autograd.grad(
                    c,
                    params,
                    retain_graph=True,
                    allow_unused=True
                )

                add_grads_(
                    g_cur,
                    grads
                )

                L_val += float(
                    c.detach()
                )

                n_kl_done += len(
                    batch_examples
                )

                del c, grads

            assert n_kl_done == N_PROTECT_FULL, (
                f"Protect KL covered {n_kl_done} "
                f"!= {N_PROTECT_FULL} samples"
            )


            # 4. HiEdit update regularizer — UNCHANGED

            L_upd = (
                gt
                * ETA
                * sum(
                    (gdW[l] ** 2).sum()
                    for l in gdW
                )
            )

            need_ref = (
                USE_AGEM
                and
                len(protect_memory) > 0
            )

            grads_upd = torch.autograd.grad(
                L_upd,
                params,
                retain_graph=need_ref,
                allow_unused=True
            )

            add_grads_(
                g_cur,
                grads_upd
            )

            L_val += float(
                L_upd.detach()
            )

            del L_upd, grads_upd


            # 5. FULL Protect-memory A-GEM — UNCHANGED scientifically

            dot = float("nan")
            ref_norm2 = 0.0
            ref_size = 0

            if need_ref:

                ref_batch = _agem_sample(
                    protect_memory,
                    AGEM_REF_BATCH
                )

                ref_size = len(
                    ref_batch
                )

                g_ref = zero_grad_buffer()

                K = len(
                    ref_batch
                )

                if K > 0:

                    chunk_size = max(
                        1,
                        AGEM_REF_CHUNK
                    )

                    starts = list(
                        range(
                            0,
                            K,
                            chunk_size
                        )
                    )

                    for j, s in enumerate(starts):

                        original_chunk = list(
                            ref_batch[
                                s:s + chunk_size
                            ]
                        )

                        microbatches = list(
                            _chunks(
                                original_chunk,
                                FAST_AGEM_MICROBATCH
                            )
                        )

                        for mb_idx, mb in enumerate(
                            microbatches
                        ):

                            L_ref_mb = (
                                nll_batch_sum(mb)
                                / K
                            )

                            is_last_original_chunk = (
                                j == len(starts) - 1
                            )

                            is_last_microbatch = (
                                mb_idx
                                ==
                                len(microbatches) - 1
                            )

                            retain = not (
                                is_last_original_chunk
                                and
                                is_last_microbatch
                            )

                            grads_ref = torch.autograd.grad(
                                L_ref_mb,
                                params,
                                retain_graph=retain,
                                allow_unused=True
                            )

                            add_grads_(
                                g_ref,
                                grads_ref
                            )

                            del (
                                L_ref_mb,
                                grads_ref
                            )


                    ref_norm2 = buffer_norm2(
                        g_ref
                    )

                    if ref_norm2 > 0.0:

                        dot = buffer_dot(
                            g_cur,
                            g_ref
                        )

                        if dot < 0.0:

                            coeff = (
                                dot
                                / ref_norm2
                            )

                            with torch.no_grad():

                                for gc, gr in zip(
                                    g_cur,
                                    g_ref
                                ):
                                    gc.add_(
                                        gr,
                                        alpha=-coeff
                                    )

                            print(
                                f"traversal {step:3d} "
                                f"| group {t:2d} "
                                f"| [A-GEM] "
                                f"memory={ref_size:3d} "
                                f"| dot={dot:.4e} < 0 "
                                f"-> PROJECTED "
                                f"(coeff={coeff:.4e})"
                            )

                        else:

                            print(
                                f"traversal {step:3d} "
                                f"| group {t:2d} "
                                f"| [A-GEM] "
                                f"memory={ref_size:3d} "
                                f"| dot={dot:.4e} >= 0 "
                                f"-> unchanged"
                            )

                    else:

                        print(
                            f"traversal {step:3d} "
                            f"| group {t:2d} "
                            f"| [A-GEM] "
                            f"||g_ref||²=0 "
                            f"-> skipped"
                        )

                else:

                    print(
                        f"traversal {step:3d} "
                        f"| group {t:2d} "
                        f"| [A-GEM] "
                        f"empty sampled memory "
                        f"-> skipped"
                    )

                del g_ref


            # 6. Accumulate projected current gradient

            with torch.no_grad():

                for acc_g, g_t in zip(
                    trajectory_grad,
                    g_cur
                ):
                    acc_g.add_(g_t)


            # 7. Remove differentiable patch

            remove_edit_patch()


            # 8. Detached sequential transition — UNCHANGED

            for l in gdW:

                previous = delta_state.get(
                    l,
                    None
                )

                if previous is None:

                    delta_state[l] = (
                        gdW[l].detach()
                    )

                else:

                    delta_state[l] = (
                        previous
                        + gdW[l].detach()
                    )


            # 9. HiEdit trajectory reward — UNCHANGED

            J += (
                gt
                * (-L_val)
            )

            groups_processed += 1

            del (
                g_cur,
                gdW,
                state_t,
                P_before_full
            )

            # no torch.cuda.empty_cache() inside group


        assert groups_processed == len(groups), (
            f"processed {groups_processed} "
            f"!= {len(groups)} groups"
        )

        opt.zero_grad(
            set_to_none=True
        )


        with torch.no_grad():

            for p, g in zip(
                params,
                trajectory_grad
            ):
                p.grad = g.clone()


        grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                editor.parameters(),
                1.0
            )
        )


        # ONE AND ONLY optimizer step

        opt.step()

        history.append(J)

        # ====================================================
        # CHECKPOINT after a COMPLETE trajectory + opt.step()
        #
        # This does not alter optimization. It only persists
        # the exact state needed to continue at step+1.
        # ====================================================

        if (
            CHECKPOINT_EVERY > 0
            and
            (
                ((step + 1) % CHECKPOINT_EVERY == 0)
                or
                (step + 1 == TRAIN_STEPS)
            )
        ):

            checkpoint_payload = {
                "editor_state_dict":
                    editor.state_dict(),

                "optimizer_state_dict":
                    opt.state_dict(),

                "history":
                    history,

                "next_step":
                    step + 1,

                "train_steps":
                    TRAIN_STEPS,

                "lr":
                    LR,

                "gamma":
                    GAMMA,

                "mu":
                    MU,

                "memory_k":
                    MEMORY_K,

                "lambda_protect":
                    LAMBDA_PROTECT,

                "eta":
                    ETA,

                "use_agem":
                    USE_AGEM,

                "agem_ref_batch":
                    AGEM_REF_BATCH,

                "agem_ref_chunk":
                    AGEM_REF_CHUNK,

                "seed":
                    SEED,

                "n_samples":
                    N_SAMPLES,

                "fast_nll_batch":
                    FAST_NLL_BATCH,

                "fast_kl_batch":
                    FAST_KL_BATCH,

                "fast_before_batch":
                    FAST_BEFORE_BATCH,

                "fast_agem_microbatch":
                    FAST_AGEM_MICROBATCH,

                **_capture_rng_state(),
            }

            _atomic_torch_save(
                checkpoint_payload,
                TRAIN_CHECKPOINT_PATH
            )

            print(
                f"[CHECKPOINT] completed traversal "
                f"{step}; next={step + 1} -> "
                f"{TRAIN_CHECKPOINT_PATH}"
            )

        print(
            f"traversal {step:3d} "
            f"| J={J:9.4f} "
            f"| grad_norm="
            f"{float(grad_norm):.4f}"
        )

        gpu_mem(
            f"traversal {step}"
        )

        del (
            trajectory_grad,
            protect_memory
        )

        if _IS_CUDA:
            torch.cuda.empty_cache()


    return history


# ============================================================
# Verification
# ============================================================

try:
    import ast
    import textwrap

    src_train = inspect.getsource(
        train_editor
    )

    # Count only REAL Python calls to opt.step(), not comments/strings.
    tree = ast.parse(
        textwrap.dedent(
            src_train
        )
    )

    opt_step_calls = 0

    for node in ast.walk(tree):

        if isinstance(
            node,
            ast.Call
        ):

            fn = node.func

            if (
                isinstance(
                    fn,
                    ast.Attribute
                )
                and
                fn.attr == "step"
                and
                isinstance(
                    fn.value,
                    ast.Name
                )
                and
                fn.value.id == "opt"
            ):
                opt_step_calls += 1

    assert (
        opt_step_calls
        == 1
    ), (
        f"Expected exactly ONE real opt.step() call, "
        f"found {opt_step_calls}"
    )

    assert "full_protect_memory" in src_train
    assert "GLOBAL_PROTECT" in src_train
    assert "AGEM_REF_BATCH" in src_train
    assert "AGEM_REF_CHUNK" in src_train
    assert "make_group_delta" in src_train

    print("assertion OK:")
    print("  FULL Protect KL        : preserved")
    print("  FULL Protect A-GEM     : preserved")
    print("  AGEM_REF_BATCH         : preserved")
    print("  AGEM_REF_CHUNK         : preserved")
    print("  FiNE/make_group_delta  : preserved")
    print(
        f"  ONE real opt.step      : preserved "
        f"(count={opt_step_calls})"
    )

except (
    OSError,
    TypeError
):
    pass


print("\nTraining configuration:")
print("  HiEdit trajectory          : ON")
print("  FULL Protect-memory KL     : ON")
print("  FULL Protect-memory A-GEM  : ON")
print("  ONE opt.step / trajectory  : ON")
print("  Scientific architecture    : UNCHANGED")
print("  FAST execution             : ON")
print("  All execution microbatches : 4")


# ============================================================
# Train editor
# ============================================================

history = train_editor()


# ============================================================
# CSC PERSISTENCE — preserved
# ============================================================

EDITOR_FINAL_PATH = run_path(
    "editor_final.pt"
)

TRAINING_HISTORY_PATH = run_path(
    "training_history.csv"
)

torch.save(
    {
        "editor_state_dict":
            editor.state_dict(),

        "history":
            history,

        "train_steps":
            TRAIN_STEPS,

        "lr":
            LR,

        "gamma":
            GAMMA,

        "mu":
            MU,

        "memory_k":
            MEMORY_K,

        "lambda_protect":
            LAMBDA_PROTECT,

        "eta":
            ETA,

        "use_agem":
            USE_AGEM,

        "agem_ref_batch":
            AGEM_REF_BATCH,

        "agem_ref_chunk":
            AGEM_REF_CHUNK,

        "seed":
            SEED,

        "n_samples":
            N_SAMPLES,

        "fast_nll_batch":
            FAST_NLL_BATCH,

        "fast_kl_batch":
            FAST_KL_BATCH,

        "fast_before_batch":
            FAST_BEFORE_BATCH,

        "fast_agem_microbatch":
            FAST_AGEM_MICROBATCH,
    },
    EDITOR_FINAL_PATH,
)

pd.DataFrame(
    {
        "traversal":
            list(
                range(
                    len(history)
                )
            ),

        "reward_J":
            history,
    }
).to_csv(
    TRAINING_HISTORY_PATH,
    index=False,
)

print(
    f"[SAVE] trained editor -> "
    f"{EDITOR_FINAL_PATH}"
)

print(
    f"[SAVE] training history -> "
    f"{TRAINING_HISTORY_PATH}"
)

# The checkpoint is intentionally kept after completion.
# It records next_step == TRAIN_STEPS and can prove the run
# finished. Delete it manually only if you want a fresh run.
if os.path.exists(TRAIN_CHECKPOINT_PATH):
    print(
        f"[CHECKPOINT] final resume file kept at -> "
        f"{TRAIN_CHECKPOINT_PATH}"
    )


# ============================================================
# Restore physical LLM to W0 after editor training
# ============================================================

remove_edit_patch()

restore_all_W0()

print(
    "\ntraining done; "
    "physical model restored to W0."
)

print(
    "Next: run the NEW Cell 9 with GENERATED classification."
)


In [ ]:
import pandas as pd

# ============================================================
# Recreate per-instance prediction buffer
# ============================================================

PER_INSTANCE_RECORDS = []


def evaluate_headline_current_model():
    """
    Evaluate CURRENT physical Qwen weights on exactly:
    CrisisLexT26 / one-shot / domain / iteration=0
    """

    PER_INSTANCE_RECORDS.clear()

    res = run_setting(
        "CrisisLexT26",
        "one",
        "domain",
        0
    )

    metrics = compute_metrics(res)

    preds = pd.DataFrame(
        PER_INSTANCE_RECORDS
    ).copy()

    return metrics, preds


In [ ]:
# ============================================================
# Cell 9 — FINAL Sequential Multi-Group Editing
#          + TRUE Generated Classification Diagnostic
#
# IMPORTANT:
#   - Cell 8 must already be trained in THIS kernel.
#   - DO NOT restart kernel.
#   - DO NOT run Cell 8 again.
#
# This cell:
#   1) restores physical model to W0
#   2) builds COMPLETE label vocabulary from ALL evaluation data
#   3) checks real generated classification at W0
#   4) performs physical sequential editing:
#
#          W0 -> W1 -> ... -> WT
#
#   5) reports BOTH:
#        - teacher-forced exact/top1
#        - REAL generated classification accuracy
#
#   6) leaves the model at WT
#
# NO optimizer.step()
# NO editor training
# ============================================================

import re
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from collections import defaultdict


# ============================================================
# 0A. Label normalization
# ============================================================

def _norm_label_text(x):

    x = str(x).strip().lower()

    # spaces / hyphens -> underscore
    x = re.sub(
        r"[\s\-]+",
        "_",
        x
    )

    # remove other punctuation
    x = re.sub(
        r"[^a-z0-9_]",
        "",
        x
    )

    # collapse repeated underscores
    x = re.sub(
        r"_+",
        "_",
        x
    )

    return x.strip("_")


# ============================================================
# 0B. Build COMPLETE canonical label vocabulary
#     from ALL evaluation samples
#
# IMPORTANT FIX:
# Do NOT derive VALID_LABELS only from groups.
# Some CrisisLexT26 classes may not appear inside Ripple/Protect groups.
# ============================================================

_label_df = pd.read_csv(
    run_path(
        "CrisisLexT26_all_predictions_per_instance.csv"
    )
).copy()


# Exact experimental scope
if "shot" in _label_df.columns:

    _label_df = _label_df[
        _label_df["shot"]
        .astype(str)
        .str.lower()
        == "one"
    ].copy()


if "prompt" in _label_df.columns:

    _label_df = _label_df[
        _label_df["prompt"]
        .astype(str)
        .str.lower()
        == "domain"
    ].copy()


if "iteration" in _label_df.columns:

    _label_df = _label_df[
        _label_df["iteration"]
        .astype(int)
        == 0
    ].copy()


assert "true_label" in _label_df.columns, (
    "Column 'true_label' not found in evaluation CSV."
)


VALID_LABELS = sorted({

    _norm_label_text(x)

    for x in (
        _label_df["true_label"]
        .dropna()
        .unique()
    )

})


assert len(VALID_LABELS) > 0, (
    "Could not reconstruct complete label vocabulary."
)


LABEL_LOOKUP = {

    _norm_label_text(label):
        label

    for label in VALID_LABELS
}


print("=" * 75)
print("COMPLETE VALID LABEL VOCABULARY")
print("=" * 75)

print(
    f"Evaluation rows used for vocabulary: "
    f"{len(_label_df)}"
)

print(
    f"Number of unique classes: "
    f"{len(VALID_LABELS)}"
)

for label in VALID_LABELS:

    print(
        " ",
        label
    )


# ============================================================
# 1. Parse generated output -> canonical class label
# ============================================================

def parse_generated_label(raw):

    raw_norm = _norm_label_text(
        raw
    )


    # --------------------------------------------------------
    # exact normalized class
    # --------------------------------------------------------

    if raw_norm in LABEL_LOOKUP:

        return LABEL_LOOKUP[
            raw_norm
        ]


    # --------------------------------------------------------
    # fallback:
    # if model generated additional text,
    # find longest valid class appearing inside output
    # --------------------------------------------------------

    matches = []


    for norm_label, canonical in LABEL_LOOKUP.items():

        if norm_label in raw_norm:

            matches.append(
                (
                    len(norm_label),
                    canonical
                )
            )


    if matches:

        matches.sort(
            reverse=True
        )

        return matches[0][1]


    return "__wrong__"


# ============================================================
# 2. TRUE generated prediction for ONE prepared example
# ============================================================

@torch.no_grad()
def generated_prediction(ex):

    prompt_ids = (
        ex["full_ids"][
            :,
            :ex["prompt_len"]
        ]
    )


    # --------------------------------------------------------
    # Generate response from CURRENT physical model
    # --------------------------------------------------------

    out = model.generate(
        input_ids=prompt_ids,
        **GEN_KWARGS
    )


    generated_ids = (
        out[
            0,
            prompt_ids.shape[1]:
        ]
    )


    raw = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )


    pred = parse_generated_label(
        raw
    )


    # --------------------------------------------------------
    # Ground-truth directly from stored target tokens
    # --------------------------------------------------------

    true_raw = tokenizer.decode(
        ex["label_ids"],
        skip_special_tokens=True
    )


    true_label = _norm_label_text(
        true_raw
    )


    return {

        "sample_id":
            str(ex["sample_id"]),

        "true_label":
            true_label,

        "predicted_label":
            pred,

        "raw_output":
            raw,

        "correct":
            bool(
                pred == true_label
            ),
    }


# ============================================================
# 3. Generated classification accuracy over examples
# ============================================================

def generated_acc_over(examples):

    if not examples:

        return {
            "acc":
                float("nan"),

            "n":
                0,

            "rows":
                []
        }


    rows = [

        generated_prediction(ex)

        for ex in examples

    ]


    acc = float(

        np.mean([

            r["correct"]

            for r in rows

        ])

    )


    return {

        "acc":
            acc,

        "n":
            len(rows),

        "rows":
            rows,
    }


# ============================================================
# 4. Prepare editor/model
# ============================================================

editor.eval()


# Remove any temporary functional patch
try:

    remove_edit_patch()

except Exception:

    pass


# ------------------------------------------------------------
# Trained editor stays trained.
# Only physical LLM weights return to W0.
# ------------------------------------------------------------

restore_all_W0()


print()

print("=" * 75)
print("W0 GENERATED-CLASSIFICATION SANITY CHECK")
print("=" * 75)


# ============================================================
# 5. W0 sanity check
# ============================================================

w0_sanity_rows = []


for t, group in enumerate(groups):

    rip_gen = generated_acc_over(
        group["members"]
    )


    prot_gen = generated_acc_over(
        group["protect"]
    )


    print(

        f"G{t:02d} "

        f"| Ripple generated="
        f"{rip_gen['acc']:.3f} "
        f"(n={rip_gen['n']}) "

        f"| Protect generated="
        f"{prot_gen['acc']:.3f} "
        f"(n={prot_gen['n']})"

    )


    w0_sanity_rows.append({

        "group":
            t,

        "group_id":
            group["gid"],

        "ripple_generated_W0":
            rip_gen["acc"],

        "n_ripple":
            rip_gen["n"],

        "protect_generated_W0":
            prot_gen["acc"],

        "n_protect":
            prot_gen["n"],
    })


w0_sanity_df = pd.DataFrame(
    w0_sanity_rows
)


W0_SANITY_PATH = run_path(
    "CrisisLexT26_W0_group_generated_sanity.csv"
)


w0_sanity_df.to_csv(
    W0_SANITY_PATH,
    index=False
)


print(
    "\nSaved:",
    W0_SANITY_PATH
)


# ============================================================
# 6. Restore W0 AGAIN before actual sequential trajectory
# ============================================================

restore_all_W0()


if (
    torch.cuda.is_available()
    and
    str(DEVICE).startswith("cuda")
):

    torch.cuda.reset_peak_memory_stats()


gpu_mem(
    "seq-edit: start"
)


# ============================================================
# 7. Sequential-edit state
# ============================================================

applied_selected = defaultdict(
    set
)


prev_members = []


seq_logs = []


generated_detail_rows = []


# ============================================================
# 8. FINAL SEQUENTIAL EDIT
#
#        W0 -> W1 -> ... -> WT
#
# NO RESET between groups.
# ============================================================

for t, group in enumerate(groups):

    print()

    print("=" * 75)

    print(
        f"[GROUP {t}] "
        f"gid={group['gid']}"
    )

    print("=" * 75)


    # ========================================================
    # A) BEFORE CURRENT EDIT — W_{t-1}
    # ========================================================

    rip_tf_before = acc_over(
        group["members"]
    )


    prot_tf_before = acc_over(
        group["protect"]
    )


    rip_gen_before = generated_acc_over(
        group["members"]
    )


    prot_gen_before = generated_acc_over(
        group["protect"]
    )


    # --------------------------------------------------------
    # P_before for local Protect KL
    # --------------------------------------------------------

    P_before = [

        answer_distribution(e)

        for e in group["protect"]

    ]


    # ========================================================
    # B) Build group ΔW under CURRENT physical W_{t-1}
    # ========================================================

    gdW = {}


    n_members = len(
        group["members"]
    )


    assert n_members > 0, (
        "final sequential edit: empty group"
    )


    for ex in group["members"]:

        # Physical model already equals W_{t-1}.
        # Therefore delta_state is empty here.

        uv = compute_uv(
            ex,
            {},
            group["edited_layers"]
        )


        with torch.no_grad():

            for l in group[
                "edited_layers"
            ]:

                d = editor.delta_w(
                    uv[l][0],
                    uv[l][1],
                    l,
                    group["col_mask"][l]
                )


                if l not in gdW:

                    gdW[l] = d

                else:

                    gdW[l] = (
                        gdW[l]
                        +
                        d
                    )


    # ========================================================
    # C) Group action MEAN
    #
    # EXACTLY ONCE
    # ========================================================

    with torch.no_grad():

        for l in gdW:

            gdW[l] = (
                gdW[l]
                /
                n_members
            )


    # ========================================================
    # D) Existing locality cap — unchanged
    # ========================================================

    gdW = cap_group_delta(
        gdW,
        group
    )


    # ========================================================
    # E) PHYSICAL EDIT
    #
    # W_{t-1} -> W_t
    # ========================================================

    with torch.no_grad():

        for l in group[
            "edited_layers"
        ]:

            w = (
                layers[l]
                .mlp
                .down_proj
                .weight
            )


            w.add_(
                gdW[l].to(
                    w.dtype
                )
            )


            applied_selected[l].update(
                group[
                    "sel_by_layer"
                ][l]
            )


    # ========================================================
    # F) AFTER CURRENT EDIT — W_t
    # ========================================================

    rip_tf_after = acc_over(
        group["members"]
    )


    prot_tf_after = acc_over(
        group["protect"]
    )


    rip_gen_after = generated_acc_over(
        group["members"]
    )


    prot_gen_after = generated_acc_over(
        group["protect"]
    )


    # ========================================================
    # G) Local Protect KL
    # ========================================================

    P_after = [

        answer_distribution(e)

        for e in group["protect"]

    ]


    if group["protect"]:

        kl = float(

            np.mean([

                F.kl_div(

                    Pa
                    .clamp_min(
                        1e-12
                    )
                    .log(),

                    Pb,

                    reduction="sum"

                ).item()

                for Pa, Pb in zip(
                    P_after,
                    P_before
                )

            ])

        )

    else:

        kl = float(
            "nan"
        )


    # ========================================================
    # H) Previous-group retention
    # ========================================================

    if prev_members:

        retention = acc_over(
            prev_members
        )

    else:

        retention = {

            "exact":
                float("nan"),

            "top1":
                float("nan"),

            "n":
                0,
        }


    # ========================================================
    # I) Existing HiEdit diagnostic losses
    # ========================================================

    with torch.no_grad():

        L_edit = float(

            sum(

                (
                    MU ** (
                        t - i
                    )
                )

                *

                float(
                    group_mean_nll(
                        groups[i]
                    )
                )

                for i in range(
                    max(
                        0,
                        t - MEMORY_K
                    ),
                    t + 1
                )

            )

        )


        L_update = float(

            sum(

                (
                    gdW[l] ** 2
                ).sum()

                for l in gdW

            )

        )


    L_t = (

        L_edit

        +

        LAMBDA_PROTECT
        *
        (
            0.0
            if math.isnan(kl)
            else kl
        )

        +

        ETA
        *
        L_update
    )


    r_low = (
        -L_t
    )


    upd_norm = float(

        math.sqrt(

            sum(

                float(
                    (
                        gdW[l] ** 2
                    ).sum()
                )

                for l in gdW

            )

        )

    )


    # ========================================================
    # J) PRINT
    # ========================================================

    print(
        f"Ripple TEACHER exact     : "
        f"{rip_tf_before['exact']:.3f}"
        f" -> "
        f"{rip_tf_after['exact']:.3f}"
    )


    print(
        f"Ripple GENERATED acc     : "
        f"{rip_gen_before['acc']:.3f}"
        f" -> "
        f"{rip_gen_after['acc']:.3f}"
        f"    (n={rip_gen_after['n']})"
    )


    print(
        f"Protect TEACHER exact    : "
        f"{prot_tf_before['exact']:.3f}"
        f" -> "
        f"{prot_tf_after['exact']:.3f}"
    )


    print(
        f"Protect GENERATED acc    : "
        f"{prot_gen_before['acc']:.3f}"
        f" -> "
        f"{prot_gen_after['acc']:.3f}"
        f"    (n={prot_gen_after['n']})"
    )


    print(
        f"Protect KL               : "
        f"{kl:.6f}"
    )


    print(
        f"Previous TEACHER top1    : "
        f"{retention['top1'] if retention['n'] else float('nan'):.3f}"
        f"    (n={retention['n']})"
    )


    print(
        f"r_low                    : "
        f"{r_low:.6f}"
    )


    print(
        f"||ΔW||                   : "
        f"{upd_norm:.6f}"
    )


    print(
        f"layers                   : "
        f"{group['edited_layers']}"
    )


    # ========================================================
    # K) Store summary
    # ========================================================

    seq_logs.append({

        "t":
            t,

        "group_id":
            group["gid"],

        "n_members":
            len(
                group["members"]
            ),

        "n_protect":
            len(
                group["protect"]
            ),


        "ripple_teacher_exact_before":
            rip_tf_before["exact"],

        "ripple_teacher_exact_after":
            rip_tf_after["exact"],

        "ripple_teacher_top1_before":
            rip_tf_before["top1"],

        "ripple_teacher_top1_after":
            rip_tf_after["top1"],


        "ripple_generated_acc_before":
            rip_gen_before["acc"],

        "ripple_generated_acc_after":
            rip_gen_after["acc"],


        "protect_teacher_exact_before":
            prot_tf_before["exact"],

        "protect_teacher_exact_after":
            prot_tf_after["exact"],


        "protect_generated_acc_before":
            prot_gen_before["acc"],

        "protect_generated_acc_after":
            prot_gen_after["acc"],


        "protect_kl":
            kl,


        "prev_retention_teacher_exact":
            retention["exact"],

        "prev_retention_teacher_top1":
            retention["top1"],

        "prev_groups_n":
            retention["n"],


        "r_low":
            r_low,

        "L_t":
            L_t,

        "L_edit":
            L_edit,

        "L_update":
            L_update,

        "update_norm":
            upd_norm,


        "edited_layers":
            ",".join(
                map(
                    str,
                    group["edited_layers"]
                )
            ),
    })


    # ========================================================
    # L) Store per-instance generated transitions
    # ========================================================

    before_by_id = {

        x["sample_id"]:
            x

        for x in (
            rip_gen_before["rows"]
            +
            prot_gen_before["rows"]
        )

    }


    after_by_id = {

        x["sample_id"]:
            x

        for x in (
            rip_gen_after["rows"]
            +
            prot_gen_after["rows"]
        )

    }


    member_ids = {

        str(
            e["sample_id"]
        )

        for e in group["members"]

    }


    protect_ids = {

        str(
            e["sample_id"]
        )

        for e in group["protect"]

    }


    for sid in sorted(

        set(
            before_by_id
        )

        |

        set(
            after_by_id
        )

    ):

        b = before_by_id.get(
            sid
        )

        a = after_by_id.get(
            sid
        )


        if (
            b is None
            or
            a is None
        ):
            continue


        if sid in member_ids:

            set_type = (
                "ripple"
            )

        elif sid in protect_ids:

            set_type = (
                "protect"
            )

        else:

            set_type = (
                "unknown"
            )


        generated_detail_rows.append({

            "t":
                t,

            "group_id":
                group["gid"],

            "set_type":
                set_type,

            "sample_id":
                sid,

            "true_label":
                b[
                    "true_label"
                ],

            "pred_before":
                b[
                    "predicted_label"
                ],

            "pred_after":
                a[
                    "predicted_label"
                ],

            "correct_before":
                b[
                    "correct"
                ],

            "correct_after":
                a[
                    "correct"
                ],

            "raw_before":
                b[
                    "raw_output"
                ],

            "raw_after":
                a[
                    "raw_output"
                ],

        })


    # ========================================================
    # M) These members become previous for next group
    # ========================================================

    prev_members.extend(
        group["members"]
    )


    # ========================================================
    # N) Cleanup
    # ========================================================

    del (
        gdW,
        P_before,
        P_after,
        rip_gen_before,
        rip_gen_after,
        prot_gen_before,
        prot_gen_after
    )


    if (
        torch.cuda.is_available()
        and
        str(DEVICE).startswith("cuda")
    ):

        torch.cuda.empty_cache()


# ============================================================
# 9. Save logs
# ============================================================

seq_logs_df = pd.DataFrame(
    seq_logs
)


SEQ_DIAG_PATH = run_path(
    "CrisisLexT26_seq_edit_generated_diagnostic.csv"
)


seq_logs_df.to_csv(
    SEQ_DIAG_PATH,
    index=False
)


generated_detail_df = pd.DataFrame(
    generated_detail_rows
)


SEQ_INSTANCE_PATH = run_path(
    "CrisisLexT26_seq_edit_generated_per_instance.csv"
)


generated_detail_df.to_csv(
    SEQ_INSTANCE_PATH,
    index=False
)


# ============================================================
# 10. Compact summary
# ============================================================

print()

print("=" * 75)

print(
    "GENERATED CLASSIFICATION SUMMARY BY GROUP"
)

print("=" * 75)


summary_cols = [

    "t",
    "group_id",
    "n_members",

    "ripple_generated_acc_before",
    "ripple_generated_acc_after",

    "n_protect",

    "protect_generated_acc_before",
    "protect_generated_acc_after",

    "protect_kl",
    "update_norm",
]


print(

    seq_logs_df[
        summary_cols
    ].to_string(
        index=False
    )

)


# ============================================================
# 11. End
# ============================================================

gpu_mem(
    "seq-edit: end"
)


print()

print(
    "Sequential editing complete: "
    "W0 -> W1 -> ... -> W%d"
    % len(groups)
)


print()

print(
    "Saved:"
)

print(
    f"  {W0_SANITY_PATH}"
)

print(
    f"  {SEQ_DIAG_PATH}"
)

print(
    f"  {SEQ_INSTANCE_PATH}"
)


print()

print("=" * 75)

print(
    "IMPORTANT: THE MODEL IS NOW AT WT."
)

print(
    "DO NOT RUN restore_all_W0() NOW."
)

print(
    "RUN YOUR AFTER EVALUATION CELL IMMEDIATELY."
)

print("=" * 75)